# KNN Customer Churn Prediction

This notebook demonstrates how to build a **K-Nearest Neighbors (KNN)** model to predict customer churn using the Telco Customer Churn dataset.

Steps:
1. Load and inspect the dataset
2. Clean and preprocess the data
3. Train a KNN model
4. Evaluate the model
5. Predict churn for new inputs

In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load the dataset
data = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

## Inspect the dataset

First, check for missing values and get a quick overview of the data.

In [55]:
data.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [56]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## Clean and prepare data

Convert `TotalCharges` to numeric (it may contain spaces), then drop rows with missing values.

In [57]:
data['TotalCharges'] = pd.to_numeric(data.TotalCharges, errors='coerce')
data.isna().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [58]:
data.dropna(inplace=True)

In [59]:
X=data.drop(['Churn', 'customerID'], axis=1)
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60


In [60]:
Y=data['Churn'].values

In [38]:
Y

<StringArray>
[ 'No',  'No', 'Yes',  'No', 'Yes', 'Yes',  'No',  'No', 'Yes',  'No',
 ...
  'No', 'Yes',  'No',  'No',  'No',  'No',  'No',  'No', 'Yes',  'No']
Length: 7032, dtype: str

## Encode categorical variables

Convert categorical columns into numeric features using one-hot encoding.

In [61]:
X.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='str')

In [62]:
X=pd.get_dummies(X,columns=['gender', 'Partner', 'Dependents',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'],drop_first=True, dtype=int)

In [63]:
X.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 30 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7032 non-null   int64  
 1   tenure                                 7032 non-null   int64  
 2   MonthlyCharges                         7032 non-null   float64
 3   TotalCharges                           7032 non-null   float64
 4   gender_Male                            7032 non-null   int64  
 5   Partner_Yes                            7032 non-null   int64  
 6   Dependents_Yes                         7032 non-null   int64  
 7   PhoneService_Yes                       7032 non-null   int64  
 8   MultipleLines_No phone service         7032 non-null   int64  
 9   MultipleLines_Yes                      7032 non-null   int64  
 10  InternetService_Fiber optic            7032 non-null   int64  
 11  InternetService_No  

## Split into training and test sets

Reserve 20% of the data for evaluation.

In [64]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2, random_state=42)

## Scale numeric features

KNN is distance-based, so scale features to the same range before training.

In [65]:
#Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled.dtype

dtype('float64')

In [66]:
X_test_scaled

array([[ 2.28524509,  1.16832097, -1.31573637, ..., -0.52764485,
        -0.71475753, -0.54742719],
       [-0.43758982, -0.54352188, -1.32567961, ..., -0.52764485,
        -0.71475753, -0.54742719],
       [-0.43758982, -0.78807086,  1.2446468 , ...,  1.89521417,
        -0.71475753, -0.54742719],
       ...,
       [-0.43758982, -0.25821474, -0.28661148, ..., -0.52764485,
        -0.71475753,  1.82672696],
       [ 2.28524509,  0.10860873,  1.52802902, ..., -0.52764485,
        -0.71475753, -0.54742719],
       [-0.43758982,  1.61666076, -1.31242196, ..., -0.52764485,
        -0.71475753, -0.54742719]], shape=(1407, 30))

In [67]:
# KNN model

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, Y_train)
Y_pred = knn.predict(X_test_scaled)
Y_pred

array(['No', 'No', 'No', ..., 'No', 'No', 'No'],
      shape=(1407,), dtype=object)

## Evaluate model performance

Compare predictions against the held-out test set using classification metrics.

In [68]:
Y_test

<StringArray>
[ 'No',  'No', 'Yes',  'No',  'No',  'No',  'No',  'No',  'No',  'No',
 ...
  'No',  'No',  'No',  'No', 'Yes',  'No',  'No',  'No',  'No',  'No']
Length: 1407, dtype: str

In [ ]:
metrics=classification_report(Y_test, Y_pred)
print(metrics)

              precision    recall  f1-score   support

          No       0.83      0.84      0.83      1033
         Yes       0.54      0.52      0.53       374

    accuracy                           0.75      1407
   macro avg       0.68      0.68      0.68      1407
weighted avg       0.75      0.75      0.75      1407



## Predict churn for a new customer

Create a new sample (feature vector must match the preprocessed data) and predict churn.

In [ ]:
#New Value prediction
data_to_test=data = [[0, 2, 87, 178, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1]]

In [52]:
data_trans=scaler.transform(data_to_test)
check=knn.predict(data_trans)
check

/opt/anaconda3/envs/py313/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array(['Yes'], dtype=object)

In [53]:
check_prob=knn.predict_proba(data_trans)
check_prob

array([[0.33333333, 0.66666667]])

## Next steps

- Tune `n_neighbors` and compare results.
- Try other models (Logistic Regression, Random Forest, etc.).
- Perform feature selection or dimensionality reduction.